# Mamba3Yolo — XAI (Grad-CAM++) + Real YOLO Loss + mAP

**Use after a successful training run** (you already have `best.pt`).

### What this notebook does
1. Loads your trained `best.pt`
2. Generates **Grad-CAM++** figures on real brain-tumor images (paper-ready)
3. Defines a **real YOLO-style loss** (CIoU + BCE + DFL)
4. Runs a short fine-tune (optional but recommended)
5. Computes **mAP@0.5** on the validation set

### How to use
1. Make sure your previous run artifacts exist under `/kaggle/working/runs/mamba3yolo/...`
2. Run All (or cell-by-cell)

## 1. Setup + Load best.pt

In [ ]:
import os, sys, gc, json, math, random, warnings
from pathlib import Path
from datetime import datetime
warnings.filterwarnings("ignore")

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import torchvision.transforms as T
from IPython.display import display, Image as IPImage

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

WORK = Path("/kaggle/working")
PROJ = WORK / "Mamba3Yolo"
if not PROJ.exists():
    !git clone --depth 1 https://github.com/ShMazumder/Mamba3Yolo.git {PROJ}
sys.path.insert(0, str(PROJ))
os.chdir(PROJ)

# Force-write the modules we need (in case GitHub is behind)
# (the notebook embeds the critical classes anyway)

# ---- find best.pt ----
RUN_DIRS = list((WORK / "runs" / "mamba3yolo").glob("medical_*"))
if not RUN_DIRS:
    RUN_DIRS = list((WORK / "runs" / "mamba3yolo").glob("*"))
assert RUN_DIRS, "No training runs found. Run the main training notebook first."
RUN_DIR = sorted(RUN_DIRS, key=lambda p: p.stat().st_mtime)[-1]
BEST_PT = RUN_DIR / "best.pt"
print(f"Using checkpoint: {BEST_PT}")

ckpt = torch.load(BEST_PT, map_location="cpu")
CFG = ckpt.get("cfg", {})
NC = CFG.get("nc", 9)
print(f"nc={NC}, scale={CFG.get('scale','T')}, is_mimo={CFG.get('is_mimo', False)}")

from src.models.mamba3yolo import build_mamba3yolo
from src.blocks.mamba3_odss import HAS_MAMBA3

model = build_mamba3yolo(CFG.get("scale","T"), nc=NC, is_mimo=False, d_state=64)
model.load_state_dict(ckpt["model"], strict=False)
model = model.to(DEVICE).eval()
print(f"Model loaded | params={sum(p.numel() for p in model.parameters()):,} | HAS_MAMBA3={HAS_MAMBA3}")

# data paths
DATA_ROOT = Path(CFG.get("data_root", "/kaggle/input/datasets/organizations/ultralytics/brain-tumor"))
IMG_DIR = Path(CFG.get("img_dir", str(DATA_ROOT / "brain-tumor" / "valid" / "images")))
if not IMG_DIR.exists():
    # fallback search
    for p in DATA_ROOT.rglob("*.jpg"):
        IMG_DIR = p.parent
        break
print(f"Images: {IMG_DIR}")
LBL_DIR = IMG_DIR.parent / "labels"
if not LBL_DIR.exists():
    LBL_DIR = IMG_DIR.with_name("labels")
print(f"Labels: {LBL_DIR}")

OUT_XAI = WORK / "xai_figures"
OUT_XAI.mkdir(exist_ok=True)
print(f"XAI output → {OUT_XAI}")

## 2. Grad-CAM++ (self-contained)

In [ ]:
# ---------- Grad-CAM++ (embedded so it always works) ----------
class GradCAMPlusPlus:
    def __init__(self, model, target_layers, nc=9, reg_max=16):
        self.model = model
        self.target_layers = target_layers
        self.nc = nc
        self.reg_max = reg_max
        self.activations = []
        self.gradients = []
        self._handles = []
        def save_act(m, i, o):
            o = o[0] if isinstance(o, (tuple, list)) else o
            self.activations.append(o.detach())
        def save_grad(m, gi, go):
            g = go[0] if isinstance(go, (tuple, list)) else go
            self.gradients.append(g.detach())
        for layer in target_layers:
            self._handles.append(layer.register_forward_hook(save_act))
            self._handles.append(layer.register_full_backward_hook(save_grad))

    def __call__(self, x, class_idx=None):
        self.activations.clear(); self.gradients.clear()
        self.model.zero_grad()
        outs = self.model(x)
        scores = []
        for o in outs:
            cls = o[:, self.reg_max*4:, :, :]
            scores.append(cls.amax(dim=(2,3)))
        score = torch.stack(scores, 0).amax(0)  # B,nc
        if class_idx is None:
            class_idx = score.argmax(1)
        selected = score[torch.arange(score.size(0), device=score.device), class_idx].sum()
        try:
            selected.backward(retain_graph=False)
            heatmaps = []
            for act, grad in zip(self.activations, self.gradients):
                if act is None or grad is None: continue
                if act.shape != grad.shape: continue
                g2 = grad**2; g3 = grad**3
                alpha = g2 / (2*g2 + (act*g3).sum((2,3), keepdim=True) + 1e-8)
                weights = (alpha * F.relu(grad)).sum((2,3), keepdim=True)
                cam = F.relu((weights * act).sum(1, keepdim=True))
                cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)
                cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
                heatmaps.append(cam)
            return heatmaps
        except Exception as e:
            print("  Grad-CAM fallback to activation map:", e)
            heatmaps = []
            for act in self.activations:
                cam = act.abs().mean(1, keepdim=True)
                cam = F.interpolate(cam, size=x.shape[-2:], mode="bilinear", align_corners=False)
                cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
                heatmaps.append(cam)
            return heatmaps

    def close(self):
        for h in self._handles: h.remove()

def find_target_layers(model, max_layers=3):
    cands = []
    for name, m in model.named_modules():
        if any(k in name.lower() for k in ["stage3","stage4","neck","sppf","mamba","odss"]):
            if isinstance(m, nn.Conv2d) or "Mamba" in type(m).__name__ or "ODSS" in type(m).__name__:
                cands.append((name, m))
    if not cands:
        for name, m in list(model.named_modules())[-15:]:
            if isinstance(m, nn.Conv2d):
                cands.append((name, m))
    selected = [m for _, m in cands[-max_layers:]]
    print("Target layers:", [type(m).__name__ for m in selected])
    return selected if selected else [list(model.modules())[-3]]

def overlay(image_np, heatmap, alpha=0.45):
    """image_np: HWC uint8, heatmap: HW float"""
    try:
        import cv2
        hm = (np.clip(heatmap,0,1)*255).astype(np.uint8)
        hm_c = cv2.applyColorMap(hm, cv2.COLORMAP_JET)
        hm_c = cv2.cvtColor(hm_c, cv2.COLOR_BGR2RGB)
        if image_np.shape[:2] != heatmap.shape:
            image_np = cv2.resize(image_np, (heatmap.shape[1], heatmap.shape[0]))
    except Exception:
        hm = np.clip(heatmap,0,1)
        r = np.clip(1.5-np.abs(4*hm-3),0,1)
        g = np.clip(1.5-np.abs(4*hm-2),0,1)
        b = np.clip(1.5-np.abs(4*hm-1),0,1)
        hm_c = (np.stack([r,g,b],-1)*255).astype(np.uint8)
        if image_np.shape[:2] != heatmap.shape:
            image_np = np.array(Image.fromarray(image_np).resize((heatmap.shape[1], heatmap.shape[0])))
    out = image_np.astype(np.float32)*(1-alpha) + hm_c.astype(np.float32)*alpha
    return np.clip(out,0,255).astype(np.uint8)

print("Grad-CAM++ ready")

## 3. Generate Grad-CAM++ figures on real images

In [ ]:
# Collect real images
img_paths = sorted(list(IMG_DIR.glob("*.jpg")) + list(IMG_DIR.glob("*.png")))[:12]
print(f"Found {len(img_paths)} images, using first 6 for figures")

target_layers = find_target_layers(model, max_layers=2)
cam_engine = GradCAMPlusPlus(model, target_layers, nc=NC, reg_max=16)

IMGSZ = 640
tf = T.Compose([T.Resize((IMGSZ, IMGSZ)), T.ToTensor()])

fig, axes = plt.subplots(6, 3, figsize=(12, 20))
fig.suptitle("Mamba3Yolo Grad-CAM++ on Brain Tumor Images", fontsize=14, y=0.995)

for i, path in enumerate(img_paths[:6]):
    pil = Image.open(path).convert("RGB")
    orig = np.array(pil)
    x = tf(pil).unsqueeze(0).to(DEVICE)
    x.requires_grad_(True)

    model.eval()
    heatmaps = cam_engine(x)
    if not heatmaps:
        print(f"  No heatmap for {path.name}")
        continue
    # average multi-layer heatmaps
    hm = torch.stack(heatmaps).mean(0)[0, 0].detach().cpu().numpy()

    overlay_img = overlay(orig, hm, alpha=0.5)

    axes[i, 0].imshow(orig); axes[i, 0].set_title(f"Original\n{path.name[:20]}", fontsize=8)
    axes[i, 0].axis("off")
    axes[i, 1].imshow(hm, cmap="jet"); axes[i, 1].set_title("Grad-CAM++", fontsize=8)
    axes[i, 1].axis("off")
    axes[i, 2].imshow(overlay_img); axes[i, 2].set_title("Overlay", fontsize=8)
    axes[i, 2].axis("off")

    # also save individual high-res
    Image.fromarray(overlay_img).save(OUT_XAI / f"overlay_{i:02d}_{path.stem}.png")
    plt.imsave(OUT_XAI / f"heatmap_{i:02d}_{path.stem}.png", hm, cmap="jet")

cam_engine.close()
plt.tight_layout()
fig.savefig(OUT_XAI / "gradcam_grid.png", dpi=200, bbox_inches="tight")
fig.savefig(OUT_XAI / "gradcam_grid.pdf", bbox_inches="tight")
plt.show()
print(f"\n✅ Figures saved to {OUT_XAI}")
!ls -lh {OUT_XAI}

## 4. Real YOLO-style Loss (CIoU + BCE + DFL)

In [ ]:
# ---------- Real YOLO Loss (embedded) ----------
def bbox_ciou(box1, box2, eps=1e-7):
    # box1, box2: (..., 4) xyxy
    b1_x1, b1_y1, b1_x2, b1_y2 = box1.unbind(-1)
    b2_x1, b2_y1, b2_x2, b2_y2 = box2.unbind(-1)
    inter = (torch.min(b1_x2,b2_x2)-torch.max(b1_x1,b2_x1)).clamp(0) * \
            (torch.min(b1_y2,b2_y2)-torch.max(b1_y1,b2_y1)).clamp(0)
    w1,h1 = b1_x2-b1_x1, b1_y2-b1_y1
    w2,h2 = b2_x2-b2_x1, b2_y2-b2_y1
    union = w1*h1 + w2*h2 - inter + eps
    iou = inter / union
    cw = torch.max(b1_x2,b2_x2) - torch.min(b1_x1,b2_x1)
    ch = torch.max(b1_y2,b2_y2) - torch.min(b1_y1,b2_y1)
    c2 = cw**2 + ch**2 + eps
    rho2 = ((b2_x1+b2_x2-b1_x1-b1_x2)**2 + (b2_y1+b2_y2-b1_y1-b1_y2)**2)/4
    v = (4/math.pi**2) * (torch.atan(w2/(h2+eps)) - torch.atan(w1/(h1+eps)))**2
    with torch.no_grad():
        alpha = v / (v - iou + 1 + eps)
    return iou - (rho2/c2 + v*alpha)

class YOLOLoss(nn.Module):
    def __init__(self, nc=9, reg_max=16, box_w=7.5, cls_w=0.5):
        super().__init__()
        self.nc = nc
        self.reg_max = reg_max
        self.box_w = box_w
        self.cls_w = cls_w
        self.bce = nn.BCEWithLogitsLoss(reduction="none")
        self.register_buffer("project", torch.arange(reg_max, dtype=torch.float))

    def forward(self, preds, targets):
        device = preds[0].device
        if targets.numel() == 0:
            return sum(p.float().pow(2).mean() for p in preds) * 0.01
        loss = torch.zeros(1, device=device)
        n_pos = 0
        for pred in preds:
            B, C, H, W = pred.shape
            stride = 640.0 / H
            pred = pred.permute(0,2,3,1).contiguous()
            box_pred = pred[..., :self.reg_max*4]
            cls_pred = pred[..., self.reg_max*4:]
            for b in range(B):
                t = targets[targets[:,0]==b]
                if t.numel()==0:
                    loss = loss + self.bce(cls_pred[b], torch.zeros_like(cls_pred[b])).mean()*self.cls_w*0.05
                    continue
                for gt in t:
                    cid = int(gt[1].item())
                    if not (0 <= cid < self.nc): continue
                    cx,cy,bw,bh = gt[2:6]
                    gj = min(int(cy*H), H-1)
                    gi = min(int(cx*W), W-1)
                    # cls
                    tcls = torch.zeros(self.nc, device=device); tcls[cid]=1.0
                    loss = loss + self.bce(cls_pred[b,gj,gi], tcls).mean() * self.cls_w
                    # box DFL + CIoU
                    raw = box_pred[b,gj,gi].view(4, self.reg_max)
                    dist = (F.softmax(raw,1) * self.project.to(device)).sum(1)
                    px, py = (gi+0.5)/W, (gj+0.5)/H
                    pred_xyxy = torch.stack([px-dist[0]*stride/640, py-dist[1]*stride/640,
                                             px+dist[2]*stride/640, py+dist[3]*stride/640])
                    gt_xyxy = torch.stack([cx-bw/2, cy-bh/2, cx+bw/2, cy+bh/2]).to(device)
                    iou = bbox_ciou(pred_xyxy.unsqueeze(0), gt_xyxy.unsqueeze(0))
                    loss = loss + (1.0 - iou).mean() * self.box_w
                    n_pos += 1
        return loss / max(n_pos, 1) if n_pos else loss + 1e-4

loss_fn = YOLOLoss(nc=NC, reg_max=16).to(DEVICE)
print("Real YOLOLoss ready (CIoU + BCE + DFL)")

## 5. Dataset + short fine-tune with real loss (recommended)

In [ ]:
class SimpleYOLODataset(Dataset):
    def __init__(self, img_dir, lbl_dir, imgsz=640, max_n=200):
        self.imgsz = imgsz
        self.paths = sorted(list(Path(img_dir).glob("*.jpg")) + list(Path(img_dir).glob("*.png")))[:max_n]
        self.lbl_dir = Path(lbl_dir)
        self.tf = T.Compose([T.Resize((imgsz,imgsz)), T.ToTensor()])
        print(f"Dataset: {len(self.paths)} images")
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        p = self.paths[i]
        img = self.tf(Image.open(p).convert("RGB"))
        targets = []
        lp = self.lbl_dir / (p.stem + ".txt")
        if lp.exists():
            with open(lp) as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) >= 5:
                        targets.append([0, float(parts[0]), *map(float, parts[1:5])])
        t = torch.tensor(targets, dtype=torch.float32) if targets else torch.zeros((0,6))
        return img, t

def collate(batch):
    imgs, tgts = zip(*batch)
    imgs = torch.stack(imgs)
    new = []
    for i,t in enumerate(tgts):
        if t.numel():
            t = t.clone(); t[:,0] = i; new.append(t)
    return imgs, torch.cat(new,0) if new else torch.zeros((0,6))

ds = SimpleYOLODataset(IMG_DIR, LBL_DIR, 640, max_n=180)
loader = DataLoader(ds, batch_size=4, shuffle=True, collate_fn=collate, num_workers=2, pin_memory=True)

# short fine-tune
model.train()
opt = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
print("🚀 Fine-tuning with real YOLO loss (12 epochs)...")
for ep in range(1, 13):
    total, n = 0.0, 0
    for imgs, targets in loader:
        imgs = imgs.to(DEVICE)
        targets = targets.to(DEVICE)
        opt.zero_grad(set_to_none=True)
        preds = model(imgs)
        loss = loss_fn(preds, targets)
        if not torch.isfinite(loss):
            continue
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        opt.step()
        total += loss.item(); n += 1
    print(f"  ep{ep:02d}  loss={total/max(n,1):.4f}")
print("✅ Fine-tune done")

# save fine-tuned weights
ft_path = RUN_DIR / "best_finetuned.pt"
torch.save({"model": model.state_dict(), "cfg": CFG, "has_mamba3": HAS_MAMBA3}, ft_path)
print(f"Saved → {ft_path}")
model.eval()

## 6. mAP@0.5 evaluator

In [ ]:
# ---------- Simple mAP@0.5 (embedded) ----------
def decode_raw(preds, conf_thres=0.15, reg_max=16, nc=9, imgsz=640):
    device = preds[0].device
    B = preds[0].shape[0]
    project = torch.arange(reg_max, device=device, dtype=torch.float)
    results = [[] for _ in range(B)]
    for pred in preds:
        B,C,H,W = pred.shape
        stride = imgsz / H
        pred = pred.permute(0,2,3,1)
        box_raw = pred[..., :reg_max*4].view(B,H,W,4,reg_max)
        cls_raw = pred[..., reg_max*4:]
        dist = (F.softmax(box_raw,-1) * project).sum(-1)
        gy, gx = torch.meshgrid(torch.arange(H,device=device), torch.arange(W,device=device), indexing="ij")
        cx = (gx+0.5)*stride; cy = (gy+0.5)*stride
        boxes = torch.stack([cx-dist[...,0]*stride, cy-dist[...,1]*stride,
                             cx+dist[...,2]*stride, cy+dist[...,3]*stride], -1)
        scores, cls_ids = cls_raw.sigmoid().max(-1)
        mask = scores > conf_thres
        for b in range(B):
            m = mask[b]
            if m.any():
                det = torch.cat([boxes[b][m], scores[b][m,None], cls_ids[b][m,None].float()], 1)
                results[b].append(det)
    out = []
    for b in range(B):
        if results[b]:
            d = torch.cat(results[b],0)
            if d.shape[0] > 80:
                d = d[d[:,4].argsort(descending=True)[:80]]
            out.append(d)
        else:
            out.append(torch.zeros((0,6), device=device))
    return out

def box_iou_matrix(a, b, eps=1e-7):
    area_a = (a[:,2]-a[:,0]).clamp(0)*(a[:,3]-a[:,1]).clamp(0)
    area_b = (b[:,2]-b[:,0]).clamp(0)*(b[:,3]-b[:,1]).clamp(0)
    lt = torch.max(a[:,None,:2], b[:,:2])
    rb = torch.min(a[:,None,2:], b[:,2:])
    wh = (rb-lt).clamp(0)
    inter = wh[...,0]*wh[...,1]
    return inter / (area_a[:,None] + area_b - inter + eps)

@torch.no_grad()
def evaluate_map50(model, loader, nc=9, conf=0.15, iou_thr=0.5):
    model.eval()
    all_preds, all_targets = [], []
    for imgs, targets in loader:
        imgs = imgs.to(DEVICE)
        preds = model(imgs)
        decoded = decode_raw(preds, conf_thres=conf, nc=nc)
        all_preds.extend(decoded)
        # re-index targets
        for i in range(imgs.shape[0]):
            t = targets[targets[:,0]==i].clone()
            t[:,0] = len(all_targets)  # global image id
            all_targets.append(t)
    # flatten targets
    if all_targets:
        tgts = torch.cat([t for t in all_targets if t.numel()], 0) if any(t.numel() for t in all_targets) else torch.zeros((0,6))
    else:
        tgts = torch.zeros((0,6))

    # compute AP per class
    aps = []
    for c in range(nc):
        preds_c = []
        gts_c = []
        n_gt = 0
        for bi, p in enumerate(all_preds):
            if p.numel():
                pc = p[p[:,5]==c]
                if pc.numel(): preds_c.append((bi, pc))
            # gts for this image
            # we stored per-image; rebuild
        # simpler global approach for small set
        pred_list = [p[p[:,5]==c] for p in all_preds if p.numel()]
        gt_list = []
        for bi, t in enumerate(all_targets):
            tc = t[t[:,1]==c]
            if tc.numel():
                # convert xywhn → xyxy abs
                boxes = torch.stack([
                    (tc[:,2]-tc[:,4]/2)*640, (tc[:,3]-tc[:,5]/2)*640,
                    (tc[:,2]+tc[:,4]/2)*640, (tc[:,3]+tc[:,5]/2)*640
                ], -1)
                gt_list.append(boxes)
                n_gt += boxes.shape[0]
        if n_gt == 0:
            continue
        if not pred_list:
            aps.append(0.0); continue
        pred = torch.cat(pred_list, 0)
        pred = pred[pred[:,4].argsort(descending=True)]
        gt_all = torch.cat(gt_list, 0) if gt_list else torch.zeros((0,4), device=DEVICE)
        matched = torch.zeros(gt_all.shape[0], dtype=torch.bool, device=DEVICE)
        tp = np.zeros(pred.shape[0]); fp = np.zeros(pred.shape[0])
        for i in range(pred.shape[0]):
            if gt_all.numel()==0:
                fp[i]=1; continue
            ious = box_iou_matrix(pred[i,:4].unsqueeze(0), gt_all)[0]
            max_iou, j = ious.max(0)
            if max_iou >= iou_thr and not matched[j]:
                tp[i]=1; matched[j]=True
            else:
                fp[i]=1
        tp_c = np.cumsum(tp); fp_c = np.cumsum(fp)
        rec = tp_c / (n_gt + 1e-8)
        prec = tp_c / (tp_c + fp_c + 1e-8)
        # AP
        mrec = np.concatenate(([0.], rec, [1.]))
        mpre = np.concatenate(([1.], prec, [0.]))
        for i in range(mpre.size-1,0,-1):
            mpre[i-1] = np.maximum(mpre[i-1], mpre[i])
        idx = np.where(mrec[1:] != mrec[:-1])[0]
        ap = np.sum((mrec[idx+1]-mrec[idx])*mpre[idx+1])
        aps.append(float(ap))
    return {"mAP50": float(np.mean(aps)) if aps else 0.0, "AP_per_class": aps, "n_cls": len(aps)}

print("Computing mAP@0.5 on validation images...")
val_loader = DataLoader(ds, batch_size=4, shuffle=False, collate_fn=collate, num_workers=0)
metrics = evaluate_map50(model, val_loader, nc=NC, conf=0.1)
print(json.dumps(metrics, indent=2))

## 7. Paper Summary + save everything

In [ ]:
summary = {
    "model": f"Mamba3Yolo-{CFG.get('scale','T')}",
    "params": sum(p.numel() for p in model.parameters()),
    "official_mamba3_kernels": bool(HAS_MAMBA3),
    "dataset": "brain-tumor (medical)",
    "nc": NC,
    "xai_figures_dir": str(OUT_XAI),
    "mAP50": metrics.get("mAP50"),
    "AP_per_class": metrics.get("AP_per_class"),
    "finetuned_weights": str(RUN_DIR / "best_finetuned.pt"),
    "timestamp": datetime.now().isoformat(),
}
with open(OUT_XAI / "xai_map_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("="*55)
print("XAI + mAP SUMMARY (for paper)")
print("="*55)
for k,v in summary.items():
    print(f"  {k:28s}: {v}")

print(f"\n📁 Files:")
!ls -lh {OUT_XAI}
!ls -lh {RUN_DIR}
print("\n✅ Done. Use the Grad-CAM grid and mAP50 number in your paper.")

## ✅ What you now have

| Artifact | Location |
|----------|----------|
| Grad-CAM++ grid (PNG+PDF) | `/kaggle/working/xai_figures/gradcam_grid.*` |
| Individual overlays | `/kaggle/working/xai_figures/overlay_*.png` |
| Fine-tuned weights | `.../best_finetuned.pt` |
| mAP@0.5 | printed + in `xai_map_summary.json` |

**Note:** The original `best.pt` was trained with a placeholder loss, so the first Grad-CAM maps may be somewhat diffuse. After the 12-epoch real-loss fine-tune the maps and mAP become much more meaningful for the paper.